# 🎯 BookVoice-AI — CosyVoice2-0.5B Test Notebook
**Stand: Juni 2026 · Python 3.11 · Drive Cache · Alle Fixes**

⚠️ Hinweis: CosyVoice2 unterstützt aktuell kein Türkisch nativ.
Cross-lingual (türkische Stimme + deutsche/englische Texte) funktioniert.

In [ ]:
#@title 💾 Schritt 0: Google Drive verbinden (einmalig)
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['MODELSCOPE_CACHE'] = '/content/drive/MyDrive/cosyvoice_models'
print('✅ Drive verbunden! Modelle werden in Drive gespeichert.')

In [ ]:
#@title ⚙️ Schritt 1: Installation + Modell laden
import sys, os

if not os.path.exists('/content/CosyVoice'):
    print('CosyVoice klonen...')
    os.system('git clone --recursive https://github.com/FunAudioLLM/CosyVoice.git')

sys.path.insert(0, '/content/CosyVoice')
sys.path.insert(0, '/content/CosyVoice/third_party/Matcha-TTS')

print('Pakete installieren...')
os.system('pip install -q -r /content/CosyVoice/requirements.txt')
os.system('pip install -q hyperpyyaml modelscope onnxruntime conformer wget openai-whisper hydra-core lightning pyworld')
print('Pakete OK!')

import torch
if not hasattr(torch, '_load_patched'):
    _orig = torch.load
    torch.load = lambda *a, **kw: _orig(*a, **{**kw, 'weights_only': False})
    torch._load_patched = True

print('Modell laden (CosyVoice2-0.5B)...')
from cosyvoice.cli.cosyvoice import CosyVoice2
model = CosyVoice2('iic/CosyVoice2-0.5B', load_jit=False, load_trt=False)
print('✅ FERTIG!')

In [ ]:
#@title 🔧 Schritt 2: Patches anwenden (immer nach Schritt 1!)
import cosyvoice.utils.file_utils as fu
import cosyvoice.cli.frontend as fe
import torchaudio, torch

def load_wav_fixed(wav, target_sr, min_sr=16000):
    if isinstance(wav, torch.Tensor):
        speech = wav.clone().float()
        if speech.shape[0] > 1:
            speech = speech.mean(dim=0, keepdim=True)
        if 16000 != target_sr:
            speech = torchaudio.functional.resample(speech, 16000, target_sr)
        return speech
    speech, sample_rate = torchaudio.load(wav)
    speech = speech.mean(dim=0, keepdim=True).float()
    if sample_rate != target_sr:
        speech = torchaudio.functional.resample(speech, sample_rate, target_sr)
    return speech

fu.load_wav = load_wav_fixed
fe.load_wav = load_wav_fixed
model.model.llm = model.model.llm.float()
print('✅ Patches OK!')

In [ ]:
#@title 🎤 Schritt 3: Stimme hochladen + generieren
from google.colab import files
from IPython.display import Audio, display
import torchaudio, whisper, torch

print('Stimme hochladen (WAV):')
uploaded = files.upload()
voice_file = list(uploaded.keys())[0]

speech, sr = torchaudio.load(voice_file)
if sr != 16000:
    speech = torchaudio.functional.resample(speech, sr, 16000)
if speech.shape[0] == 2:
    speech = speech.mean(dim=0, keepdim=True)
speech_8s = speech[:, :16000*8].float()
torchaudio.save('/content/stimme_8s.wav', speech_8s, 16000)
print(f'✅ Stimme: {speech_8s.shape[1]/16000:.1f} Sek, Mono, 16kHz')

print('Transkribiere...')
w = whisper.load_model('tiny')
result = w.transcribe('/content/stimme_8s.wav')
prompt_text = result['text'].strip()
print(f'Prompt Text: {prompt_text}')

# Zieltext — unterstützte Sprachen: DE/EN/ZH/JA/KO
ziel_text = 'Merhaba, benim adım Ahrar. Ben bir Türk yazarıyım.' #@param {type:"string"}

print('Generiere...')
for i, res in enumerate(model.inference_zero_shot(
    ziel_text,
    prompt_text,
    speech_8s,
    stream=False
)):
    torchaudio.save('/content/cosy_output.wav', res['tts_speech'], model.sample_rate)
    print('✅ Fertig!')
    display(Audio('/content/cosy_output.wav'))

In [ ]:
#@title ⏳ Schritt 4: Session aktiv halten — IMMER laufen lassen!
import time
print('Session läuft... (nicht stoppen!)')
while True:
    time.sleep(60)
    print('.', end='', flush=True)